---
last_verified: 2026-10-28
tool_version: n/a
sources: []
---

# Querying Prometheus with PromQL — step-through

A walk through the PromQL query language against the Prometheus HTTP API: instant queries, range queries, functions, operators, and the recording and alerting rules that consume the results.

In [ ]:
import json, urllib.parse, urllib.request

PROM = "http://localhost:9090"

def q(expr):
    url = f"{PROM}/api/v1/query?query={urllib.parse.quote(expr)}"
    with urllib.request.urlopen(url, timeout=10) as r:
        return json.load(r)

print(json.dumps(q("up"), indent=2)[:400])

In [ ]:
# Range query: per-second rate of http_requests_total over 5m buckets
url = f"{PROM}/api/v1/query_range?query=rate(http_requests_total[5m])&start=now-1h&end=now&step=60"
with urllib.request.urlopen(url, timeout=10) as r:
    data = json.load(r)
print(len(data['data']['result']), "series returned")

In [ ]:
# Aggregation: sum by job, averaged over the last hour
expr = "avg_over_time(http_request_duration_seconds_bucket[1h])"
print(q(expr)['data']['result'][0]['metric'])

In [ ]:
# Alerting rule check: is the high-error-rate rule firing?
print(q("ALERTS_FOR_STATE").get('data', {}).get('resultCount', 'n/a'))